In [3]:
# add parent to path
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent.parent))
from utils.logging_and_reporting.wandb_api_helper import wandb_retrieve_metrics_for_run

wandb_run_id = "m67to2p5"
statistics, config, hist = wandb_retrieve_metrics_for_run(
    benchmark="tpch", run_id=wandb_run_id, output_hist=True
)

In [4]:
statistics

{'code/snapshot_hash': '44acfe9d44f8ee188fa8722acd5a2a5d7daa25e3',
 'scalefactor': None,
 'query_runtimes': None}

In [5]:
import pandas as pd

from demo_and_analysis.plots.utils.wandb_trace_preprocessor import DataCleaner

assert isinstance(hist, pd.DataFrame), "Expected hist to be a DataFrame"

snapshot_col = "code/snapshot_hash"
if snapshot_col not in hist.columns:
    raise ValueError(f"Missing required column: {snapshot_col}")

# Reuse the same stage extraction logic as plot_timeline.py.
worked_on = DataCleaner.extract_worked_on_queries(hist, drill_down_to_query_level=False)
spans = DataCleaner.extract_worked_on_spans(worked_on)

stage_last_hash = {}
for span in spans:
    stage = span.section or "unknown"
    stage_slice = hist.iloc[span.start : span.end + 1]
    valid_hashes = stage_slice[snapshot_col].dropna()
    if not valid_hashes.empty:
        stage_last_hash[stage] = valid_hashes.iloc[-1]

if not stage_last_hash:
    print("No snapshot hashes found in history.")
else:
    print("Final snapshot hash per stage:")
    for stage, snap in stage_last_hash.items():
        print(f"- {stage}: {snap}")

# Optional convenience handle for downstream cells.
final_snapshot_hash = next(reversed(stage_last_hash.values()), None)
print(f"\nFinal stage snapshot hash: {final_snapshot_hash}")

Final snapshot hash per stage:
- storage: b5631320b846eeee478214554e6fd342f10bcdf0
- pin & trace: d0aeb35219b8113d16bdff15b956424361b71908
- optim card: a1c081243a72b365cabea0c24f47078debe2ef47
- optim trace: 5ecfd6a3755cc62749b9acfa5966f6d44715f1b7
- optim expert: 5e2bd438c17b5800feb3daeebab1430846a5ecf5
- optim human: 44acfe9d44f8ee188fa8722acd5a2a5d7daa25e3

Final stage snapshot hash: 44acfe9d44f8ee188fa8722acd5a2a5d7daa25e3
